# PAP Emulation - ngspice Circuit Transient

Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary. Licensed under GNU GPL v3.0. Third-party dependencies retain their own licenses.

A native ngspice RC transient simulation validated against the analytical first-order circuit response and exported through the common PAP result contract.

**Author:** Abhishek Choudhary · **Affiliation:** AyeAI · **ORCID:** [0009-0002-0684-8320](https://orcid.org/0009-0002-0684-8320)

**Deployment:** browser-only platform; this notebook runs in a user-controlled Colab runtime. GPU/TPU availability is variable and not guaranteed. **Evidence boundary:** results remain `SIMULATED` and `UNVERIFIED`; this is not HIL or physical verification.

## 0. Runtime, security and provenance
Run cells in order. The notebook records backend versions, solver settings, runtime device, model configuration, timestamps and SHA-256 job/result digests. Results must be imported through the separate PAP Colab Bridge. Never upload untrusted native FMUs or binaries.

In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.
# AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
# Embedded in each notebook so notebooks remain independently runnable.
PAP_BRIDGE_VERSION = '1.0.0'
PAP_JOB_SCHEMA = 'pap.emulation.job/1.0'
PAP_RESULT_SCHEMA = 'pap.emulation.result/1.0'

def pap_canonical(value):
    import json
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False, allow_nan=False)

def pap_digest(text):
    import hashlib
    return 'sha256:' + hashlib.sha256(text.encode('utf-8')).hexdigest()

def pap_make_job(backend_id, operation, inputs):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_JOB_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'createdAt': datetime.now(timezone.utc).isoformat(),
        'target': {'backendId': backend_id, 'backendVersion': None, 'operation': operation, 'requestedDevice': 'auto'},
        'sir': {'header': {'id': 'colab-reference-fixture', 'irVersion': '1.0.0'}, 'components': []},
        'inputs': inputs,
        'observations': None,
        'configuration': {},
        'provenance': {'source': 'COLAB_REFERENCE_FIXTURE', 'sourceVersion': '1.0.0', 'license': 'GPL-3.0-only'}
    }
    canonical = pap_canonical(body)
    digest = pap_digest(canonical)
    return {**body, 'jobId': 'papjob-' + digest[7:27], 'digest': digest, 'canonicalPayload': canonical}

def pap_verify_job(job, expected_backend):
    import json
    if not isinstance(job, dict) or job.get('schema') != PAP_JOB_SCHEMA:
        raise ValueError('Unsupported/missing PAP job schema')
    payload = job.get('canonicalPayload')
    if not isinstance(payload, str): raise ValueError('Job canonicalPayload missing')
    parsed = json.loads(payload)
    body = {k:v for k,v in job.items() if k not in ('jobId','digest','canonicalPayload')}
    if pap_canonical(parsed) != pap_canonical(body): raise ValueError('Job payload/body mismatch')
    digest = pap_digest(payload)
    if digest != job.get('digest') or job.get('jobId') != 'papjob-' + digest[7:27]:
        raise ValueError('Job digest/jobId verification failed')
    if job.get('target',{}).get('backendId') != expected_backend:
        raise ValueError('Job targets a different backend')
    if not isinstance(job.get('inputs'), dict): raise ValueError('Job inputs must be an object')
    return job

def pap_get_job(backend_id, operation, default_inputs, use_demo=True):
    if use_demo:
        return pap_make_job(backend_id, operation, default_inputs), True
    from google.colab import files
    import json
    uploaded = files.upload()
    candidates = [name for name in uploaded if name.lower().endswith('.json')]
    if len(candidates) != 1: raise ValueError('Upload exactly one browser-exported PAP job .json')
    return pap_verify_job(json.loads(uploaded[candidates[0]].decode('utf-8')), backend_id), False

def pap_make_result(job, backend_id, backend_version, outputs, diagnostics, device='cpu'):
    from datetime import datetime, timezone
    body = {
        'schema': PAP_RESULT_SCHEMA,
        'bridgeVersion': PAP_BRIDGE_VERSION,
        'jobId': job['jobId'],
        'jobDigest': job['digest'],
        'backend': {'id': backend_id, 'version': str(backend_version)},
        'execution': {'status': 'EXECUTED', 'runtime': 'COLAB', 'startedAt': None,
                      'finishedAt': datetime.now(timezone.utc).isoformat(), 'device': str(device)},
        'outputs': outputs,
        'diagnostics': diagnostics,
        'fidelity': 'SIMULATED',
        'physicalVerification': False,
        'evidenceState': 'UNVERIFIED',
        'claims': {'verified': False, 'hil': False, 'physical': False}
    }
    canonical = pap_canonical(body)
    return {**body, 'canonicalPayload': canonical, 'resultDigest': pap_digest(canonical)}

def pap_export(job, result, is_demo, result_filename='pap-emulation-result.json'):
    import json
    with open(result_filename, 'w', encoding='utf-8') as stream:
        json.dump(result, stream, indent=2, ensure_ascii=False, allow_nan=False)
    if is_demo:
        with open('pap-emulation-job.json', 'w', encoding='utf-8') as stream:
            json.dump(job, stream, indent=2, ensure_ascii=False, allow_nan=False)
    try:
        from google.colab import files
        files.download(result_filename)
        if is_demo: files.download('pap-emulation-job.json')
    except ImportError:
        print('Saved locally:', result_filename, '(download manually outside Colab)')
    print('Result remains SIMULATED / UNVERIFIED. Import it through the PAP Colab Bridge.')


In [ ]:
# Copyright © 1993-2026 Abhishek Choudhary. All rights reserved. AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.
# SPDX-License-Identifier: GPL-3.0-only
!apt-get -qq update && apt-get -qq install -y ngspice

In [ ]:
import subprocess, pathlib, math, json, hashlib
version=subprocess.run(['ngspice','--version'],capture_output=True,text=True,check=True).stdout
print(version.splitlines()[:8])
PAP_USE_DEMO_JOB=True
default_inputs={'resistanceOhm':1000.0,'capacitanceFarad':1e-6,'stopTimeSeconds':0.005,'sampleStepSeconds':5e-5}
job,job_is_demo=pap_get_job('ngspice','ngspice.rc_transient',default_inputs,PAP_USE_DEMO_JOB)

## 1. RC transient analytical benchmark
A 1 V step drives R and C. The ideal capacitor response is `1-exp(-t/(RC))`. The 6% error bound here is a coarse smoke-test threshold accounting for the idealized source edge and sampled transient; it is not a general SPICE accuracy qualification.

In [ ]:
R=float(job['inputs'].get('resistanceOhm',1000)); Cval=float(job['inputs'].get('capacitanceFarad',1e-6)); stop=float(job['inputs'].get('stopTimeSeconds',0.005)); step=float(job['inputs'].get('sampleStepSeconds',5e-5)); tau=R*Cval
if min(R,Cval,stop,step)<=0 or stop/step>2_000_000: raise ValueError('Invalid or excessive RC request')
netlist=f'''PAP RC transient validation\nVSTEP in 0 PULSE(0 1 0 1n 1n 10 20)\nR1 in out {R}\nC1 out 0 {Cval}\n.control\nset noaskquit\ntran {step} {stop}\nwrdata rc-transient.dat time v(out)\nquit\n.endc\n.end\n'''
pathlib.Path('pap_rc.cir').write_text(netlist)
run=subprocess.run(['ngspice','-b','pap_rc.cir'],capture_output=True,text=True)
if run.returncode!=0: raise RuntimeError(run.stdout+'\n'+run.stderr)
data=[]
for line in pathlib.Path('rc-transient.dat').read_text().splitlines():
 vals=[float(x) for x in line.split()]
 if len(vals)>=2 and all(math.isfinite(x) for x in vals): data.append((vals[0],vals[-1]))
if len(data)<10: raise RuntimeError('Insufficient transient samples')
errors=[abs(v-(1-math.exp(-t/tau))) for t,v in data if t>=0]; max_error=max(errors)
print({'samples':len(data),'maxAbsError':max_error,'returncode':run.returncode}); assert max_error<0.06

In [ ]:
outputs=[{'sequence':int(i),'time':float(t),'values':{'capacitorVoltage_V':float(v)}} for i,(t,v) in enumerate(data)]
result=pap_make_result(job,'ngspice',version.splitlines()[0] if version.splitlines() else 'unknown',outputs,{'R_ohm':R,'C_farad':Cval,'timeConstantSeconds':tau,'maxAbsErrorAgainstAnalytical':max_error,'sampleCount':len(data),'mappingRequired':True},'cpu')
pap_export(job,result,job_is_demo,'pap-ngspice-result.json')